# Research source snapshot
Outputs and execution metadata have been removed. Read the repository README before running. These exploratory notebooks are not validated end to end. Only load trusted local pickle files.


In [ ]:
import os
from pathlib import Path

REPOSITORY_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_ROOT = Path(os.environ.get("RESEARCH_DATA_DIR", str(REPOSITORY_ROOT / "data"))).expanduser().resolve()
OUTPUT_ROOT = Path(os.environ.get("RESEARCH_OUTPUT_DIR", str(REPOSITORY_ROOT / "outputs"))).expanduser().resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
import pickle
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import numpy as np
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
import os

 **ACCESS BEST COLUMNS OBTAINED FROM 2011-2001 DATA**

In [ ]:
# Specify the path to the pickle file
best_columns_file_path = str(DATA_ROOT / "top_selected_feature_using_mean.pickle")

# Open the pickle file in read mode
with open(best_columns_file_path, "rb") as f:
    # Load the contents of the pickle file
    cols_best = pickle.load(f)

In [ ]:
cols_best

In [ ]:
data1 = pickle.load(open(str(DATA_ROOT / 'dataset.pickle'), 'rb'))

In [ ]:
data1

In [ ]:
# Extract X_train, y_train, X_test, y_test
X_train = data1['X_train']
X_test = data1['X_test']

In [ ]:
X_train.index

In [ ]:
X_test.index

In [ ]:
import pickle
with open(str(OUTPUT_ROOT / "new_relative_wealth_index.pickle"), "rb") as f:
    rwi_data = pickle.load(f)

In [ ]:
rwi_data.index

In [ ]:
rwi_data.head(20)

In [ ]:
# Check if indexes of X_train and X_test are present in rwi_data index
train_indexes_present = X_train.index.isin(rwi_data.index).all()
test_indexes_present = X_test.index.isin(rwi_data.index).all()

print("Are all indexes of X_train present in rwi_data index?", train_indexes_present)
print("Are all indexes of X_test present in rwi_data index?", test_indexes_present)

In [ ]:
# Get the indexes of X_train and X_test
train_indexes = X_train.index
test_indexes = X_test.index

# Subset the rwi_data DataFrame using the indexes
rwi_data_train = rwi_data.loc[X_train.index]
rwi_data_test = rwi_data.loc[X_test.index]

# Create new y_train and y_test based on the subsetted DataFrame
y_train = rwi_data_train['RWI']
y_test = rwi_data_test['RWI']

In [ ]:
X_train_best = X_train[cols_best]
X_test_best = X_test[cols_best]

In [ ]:
best_params = {
    'max_features': 'sqrt',
    'max_samples': 0.7,
    'min_samples_split': 75,
    'n_estimators': 80,
    'n_jobs': -1,
    'oob_score': True,
    'random_state': 0
    }

In [ ]:
# Initialize and train the RandomForestRegressor model
rf = RandomForestRegressor(**best_params)
rf.fit(X_train_best, y_train)

In [ ]:
predictions_with_best_columns=rf.predict(X_test_best)

In [ ]:
import matplotlib.pyplot as plt

# Assuming predictions is the array of predicted values
# Plot the scatter plot
plt.figure(figsize=(8, 6))
plt.scatter(y_test, predictions_with_best_columns, color='blue', alpha=0.5)
plt.title('Scatter Plot of Predicted vs Actual Values')
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.grid(True)
plt.show()

In [ ]:
# Save the model to a file using pickle
with open(str(OUTPUT_ROOT / 'random_forest_model.pickle'), 'wb') as f:
    pickle.dump(rf, f)

In [ ]:
# Load the model from the file using pickle
with open(str(OUTPUT_ROOT / 'random_forest_model.pickle'), 'rb') as f:
    loaded_model = pickle.load(f)

In [ ]:
# Once trained, you can use the model for predictions
predictions = loaded_model.predict(X_test_best)

In [ ]:
import matplotlib.pyplot as plt

# Assuming predictions is the array of predicted values
# Plot the scatter plot
plt.figure(figsize=(8, 6))
plt.scatter(y_test, predictions, color='blue', alpha=0.5)
plt.title('Scatter Plot of Predicted vs Actual Values')
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.grid(True)
plt.show()

In [ ]:
from sklearn.metrics import mean_squared_error
import numpy as np

rmse = np.sqrt(mean_squared_error(y_test, predictions))
print("Root Mean Squared Error (RMSE):", rmse)

In [ ]:
# Assuming rmse is the RMSE value and y_train contains the target variable

# Calculate the range of the target variable
target_range = y_train.max() - y_train.min()

# Calculate normalized RMSE value
normalized_rmse = rmse / target_range

# Display normalized RMSE value
print("Normalized RMSE:", normalized_rmse)

In [ ]:
# Step 1: Calculate the mean of the target variable
target_mean = y_train.mean()

In [ ]:
target_mean

In [ ]:
# Step 3: Normalize the RMSE by dividing it by the mean of the target variable
normalized_rmse = rmse / target_mean

print("Normalized RMSE by mean:", normalized_rmse)

In [ ]:
from sklearn.metrics import mean_squared_error
import numpy as np

# Filter predictions and actual values for values between -1 to 1 and other values
mask_1_to_minus_1 = (y_test >= -1) & (y_test <= 1)
predictions_1_to_minus_1 = predictions[mask_1_to_minus_1]
y_test_1_to_minus_1 = y_test[mask_1_to_minus_1]

predictions_other = predictions[~mask_1_to_minus_1]
y_test_other = y_test[~mask_1_to_minus_1]

# Calculate RMSE for values between -1 to 1
rmse_1_to_minus_1 = np.sqrt(mean_squared_error(y_test_1_to_minus_1, predictions_1_to_minus_1))

# Calculate RMSE for other values
rmse_other = np.sqrt(mean_squared_error(y_test_other, predictions_other))

print("RMSE for values between -1 to 1:", rmse_1_to_minus_1)
print("RMSE for other values:", rmse_other)